# DeepFace Online Endpoint Deployment (Azure ML)

## Readfirst
This notebook deploys (or updates) a **Managed Online Endpoint** in Azure Machine Learning for DeepFace scoring.
It is **idempotent**: it checks whether assets already exist (model / environment / endpoint / deployment) and only creates them when missing.

### Prerequisites
- Install deps (must include `azure-ai-ml` and `azure-identity`).
- Authenticate with Azure: set `AZURE_TENANT_ID`, `AZURE_CLIENT_ID`, `AZURE_CLIENT_SECRET` in .dotenv
- Set workspace env vars in .dotenv: `AZURE_SUBSCRIPTION_ID`, `AZURE_RESOURCE_GROUP`, `AZUREML_WORKSPACE_NAME`.

### How to retrieve scoring URI + secret
Run all cells in this notebook. The cell in section "Get scoring URI and key" will print the scoring URI and secret key to use when calling the endpoint.

### Delete the endpoint
You have to delete the endpoint manually with the Azure ML SDK after you are done testing it, to avoid incurring ongoing costs. Setting up the CONFIRM_DELETE variable to True and running the last cell in this notebook will delete the endpoint.


# 1) Imports and configuration

In [1]:
import os
import json
import base64
import urllib.request
from pathlib import Path
from dotenv import load_dotenv

from azure.identity import ClientSecretCredential
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Environment,
    CodeConfiguration,
    OnlineRequestSettings,
    Model,
)
from azure.core.exceptions import ResourceNotFoundError

# Load environment variables from .env file
load_dotenv()

True

In [2]:
# Authentication 
TENANT_ID = os.getenv("AZURE_TENANT_ID")
CLIENT_ID = os.getenv("AZURE_CLIENT_ID")
CLIENT_SECRET = os.getenv("AZURE_CLIENT_SECRET")
if not all([TENANT_ID, CLIENT_ID, CLIENT_SECRET]):
    raise EnvironmentError("Missing required env vars for authentication: AZURE_TENANT_ID, AZURE_CLIENT_ID, AZURE_CLIENT_SECRET")

# Required workspace targeting (NO secrets here)
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID")
RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP")
WORKSPACE_NAME = os.getenv("AZUREML_WORKSPACE_NAME")

missing = [name for name, val in [
    ("AZURE_SUBSCRIPTION_ID", SUBSCRIPTION_ID),
    ("AZURE_RESOURCE_GROUP", RESOURCE_GROUP),
    ("AZUREML_WORKSPACE_NAME", WORKSPACE_NAME),
] if not val]
if missing:
    raise EnvironmentError(f"Missing required env vars: {', '.join(missing)}")

# Endpoint naming (per requirement)
ENDPOINT_NAME = "deep-face-endpoint"

# AML asset naming (adjust versions if you need a new revision)
MODEL_NAME = "deepface-models"
MODEL_VERSION = "1"
ENV_NAME = "deepface-online-endpoint-environment"
ENV_VERSION = "1"
DEPLOYMENT_NAME = "blue"

# Local folders in this repo
THIS_DIR = Path.cwd()
SCORING_CODE_DIR = THIS_DIR / "DeepFace" / "ScoringScript"
CONDA_FILE = THIS_DIR / "DeepFace" / "Environment" / "conda.yaml"

# 2) Connect to Azure ML Workspace
We use `DefaultAzureCredential` so secrets are never hardcoded in the notebook.

In [4]:
credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
) 

ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print("Connected to workspace:", WORKSPACE_NAME)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Connected to workspace: ws-coursera-azure


# 3) Get-or-create the DeepFace weights model asset
This registers a folder structured like `.deepface/weights/<weight_file>` as an Azure ML model named `deepface-models`.

In [11]:
def try_get_model(name: str, version: str):
    try:
        return ml_client.models.get(name=name, version=version)
    except ResourceNotFoundError:
        return None

def ensure_deepface_model() -> Model:
    existing = try_get_model(MODEL_NAME, MODEL_VERSION)
    if existing is not None:
        print(f"Model exists: {MODEL_NAME}:{MODEL_VERSION}")
        return existing

    print(f"Model not found. Creating: {MODEL_NAME}:{MODEL_VERSION}")
    tmp_root = THIS_DIR / "temporary" / ".deepface" / "weights"
    tmp_root.mkdir(parents=True, exist_ok=True)

    facenet_url = "https://github.com/serengil/deepface_models/releases/download/v1.0/facenet512_weights.h5"
    retinaface_url = "https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5"

    facenet_path = tmp_root / "facenet512_weights.h5"
    retinaface_path = tmp_root / "retinaface.h5"

    if not facenet_path.exists():
        urllib.request.urlretrieve(facenet_url, str(facenet_path))
    if not retinaface_path.exists():
        urllib.request.urlretrieve(retinaface_url, str(retinaface_path))

    model_asset = Model(
        name=MODEL_NAME,
        version=MODEL_VERSION,
        path=str((THIS_DIR / "temporary" / ".deepface")),
        description="DeepFace model weights folder (.deepface/weights)",
    )
    created = ml_client.models.create_or_update(model_asset)
    print(f"Created model: {created.name}:{created.version}")
    
    # Clean up temporary files
    try:
        facenet_path.unlink()
        retinaface_path.unlink()
        tmp_root.rmdir()
    except Exception as e:
        print(f"Warning: could not clean up temporary files: {e}")

    return created

deepface_model = ensure_deepface_model()

Model exists: deepface-models:1


# 4) Get-or-create the environment
This environment must contain your runtime dependencies (from `DeepFaceOnlineScoring/Environment/conda.yaml`).

In [12]:
def try_get_environment(name: str, version: str):
    try:
        return ml_client.environments.get(name=name, version=version)
    except ResourceNotFoundError:
        return None

def ensure_environment() -> Environment:
    existing = try_get_environment(ENV_NAME, ENV_VERSION)
    if existing is not None:
        print(f"Environment exists: {ENV_NAME}:{ENV_VERSION}")
        return existing

    print(f"Environment not found. Creating: {ENV_NAME}:{ENV_VERSION}")
    env = Environment(
        name=ENV_NAME,
        version=ENV_VERSION,
        description="Environment for DeepFace online endpoint",
        conda_file=str(CONDA_FILE),
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    )
    created = ml_client.environments.create_or_update(env)
    print(f"Created environment: {created.name}:{created.version}")
    return created

deepface_env = ensure_environment()

Environment exists: deepface-online-endpoint-environment:1


# 5) Get-or-create the endpoint
Endpoint name is fixed to **deep-face-endpoint** and uses `auth_mode=key`.

In [13]:
def try_get_endpoint(name: str):
    try:
        return ml_client.online_endpoints.get(name=name)
    except ResourceNotFoundError:
        return None

def ensure_endpoint() -> ManagedOnlineEndpoint:
    existing = try_get_endpoint(ENDPOINT_NAME)
    if existing is not None:
        print(f"Endpoint exists: {ENDPOINT_NAME}")
        return existing

    print(f"Endpoint not found. Creating: {ENDPOINT_NAME}")
    endpoint = ManagedOnlineEndpoint(
        name=ENDPOINT_NAME,
        description="DeepFace scoring endpoint",
        auth_mode="key",
    )
    poller = ml_client.online_endpoints.begin_create_or_update(endpoint)
    created = poller.result()
    print(f"Created endpoint: {created.name}")
    return created

endpoint = ensure_endpoint()

Endpoint not found. Creating: deep-face-endpoint
Created endpoint: deep-face-endpoint


# 6) Get-or-create the deployment
We create a single deployment named `blue` and route 100% traffic to it.

In [14]:
def try_get_deployment(endpoint_name: str, deployment_name: str):
    try:
        return ml_client.online_deployments.get(name=deployment_name, endpoint_name=endpoint_name)
    except ResourceNotFoundError:
        return None

def ensure_deployment() -> ManagedOnlineDeployment:
    existing = try_get_deployment(ENDPOINT_NAME, DEPLOYMENT_NAME)
    if existing is not None:
        print(f"Deployment exists: {DEPLOYMENT_NAME} (endpoint={ENDPOINT_NAME})")
        return existing

    print(f"Deployment not found. Creating: {DEPLOYMENT_NAME} (endpoint={ENDPOINT_NAME})")
    deployment = ManagedOnlineDeployment(
        name=DEPLOYMENT_NAME,
        endpoint_name=ENDPOINT_NAME,
        model=deepface_model,
        environment=deepface_env,
        code_configuration=CodeConfiguration(
            code=str(SCORING_CODE_DIR),
            scoring_script="score.py",
        ),
        instance_type="Standard_F2s_v2",
        instance_count=1,
        request_settings=OnlineRequestSettings(
            max_concurrent_requests_per_instance=1,
            request_timeout_ms=30000,
        ),
    )
    poller = ml_client.online_deployments.begin_create_or_update(deployment)
    created = poller.result()
    print(f"Created deployment: {created.name}")
    return created

deployment = ensure_deployment()

Check: endpoint deep-face-endpoint exists


Deployment not found. Creating: blue (endpoint=deep-face-endpoint)
.........................................................................Created deployment: blue


In [15]:
# Route 100% of traffic to the deployment (safe to re-run)
endpoint = ml_client.online_endpoints.get(name=ENDPOINT_NAME)
endpoint.traffic = {DEPLOYMENT_NAME: 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Traffic set to", endpoint.traffic)

Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>
Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


Traffic set to {'blue': 100}


# 7) Get scoring URI and API key (Azure ML SDK)
This is the canonical way to fetch the endpoint URL and secret without hardcoding.

In [16]:
endpoint = ml_client.online_endpoints.get(name=ENDPOINT_NAME)
keys = ml_client.online_endpoints.get_keys(name=ENDPOINT_NAME)

print("Endpoint name:", endpoint.name)
print("Scoring URI:", endpoint.scoring_uri)
print("Primary key (secret):", keys.primary_key)

Endpoint name: deep-face-endpoint
Scoring URI: https://deep-face-endpoint.southeastasia.inference.ml.azure.com/score
Primary key (secret): EJcpwHkDD89jzoaIOfWHSPnQ85uHSp6gwxw5QgAiGoG2aQxAwObyJQQJ99BLAAAAAAAAAAAAINFRAZML22Z7


# 8) Invoke the endpoint
Choose either SDK invocation (recommended) or direct REST call (useful outside AML).

## 8.1 Create a sample request JSON
Set `IMAGE_PATH` to a local image file. This creates `sample_request.json` with a base64-encoded image.

In [ ]:
# IMAGE_PATH = os.getenv("DEEPFACE_SAMPLE_IMAGE", "")  # optional env var to avoid editing the notebook
# if not IMAGE_PATH:
#     print("Set DEEPFACE_SAMPLE_IMAGE env var (or assign IMAGE_PATH) to create a request file.")
# else:
#     image_path = Path(IMAGE_PATH)
#     if not image_path.exists():

#     with open(image_path, "rb") as f:
#         image_base64 = base64.b64encode(f.read()).decode("utf-8")
#     payload = {"image": image_base64}
#     with open(THIS_DIR / "sample_request.json", "w", encoding="utf-8") as f:
#         json.dump(payload, f)
#     print("Wrote sample_request.json")

## 8.2 Invoke via Azure ML SDK

In [ ]:
# request_file = str(THIS_DIR / "sample_request.json")
# if not Path(request_file).exists():

# result = ml_client.online_endpoints.invoke(
#     endpoint_name=ENDPOINT_NAME,
#     deployment_name=DEPLOYMENT_NAME,
#     request_file=request_file,
# )
# print(result)

## 8.3 Invoke via REST (outside Azure ML)
This uses the scoring URI + primary key retrieved above.

In [ ]:
# import requests

# endpoint = ml_client.online_endpoints.get(name=ENDPOINT_NAME)
# keys = ml_client.online_endpoints.get_keys(name=ENDPOINT_NAME)

# scoring_uri = endpoint.scoring_uri
# api_key = keys.primary_key

# with open(THIS_DIR / "sample_request.json", "r", encoding="utf-8") as f:
#     payload = json.load(f)

# headers = {
#     "Content-Type": "application/json",
#     "Authorization": f"Bearer {api_key}",
# }
# resp = requests.post(scoring_uri, json=payload, headers=headers, timeout=60)
# print("Status:", resp.status_code)
# print(resp.text)

# 9) Delete the endpoint (cleanup)
Deleting the endpoint stops billing for the managed online endpoint + deployment.

**Safety:** the code below is guarded; you must explicitly set `CONFIRM_DELETE=True` for it to run.

In [ ]:
# DANGER ZONE: Cleanup / delete endpoint
from azure.core.exceptions import ResourceNotFoundError

# Set this to True to actually delete resources.
CONFIRM_DELETE = False

if not CONFIRM_DELETE:
    raise RuntimeError(
        f"Refusing to delete endpoint '{ENDPOINT_NAME}'. Set CONFIRM_DELETE=True to proceed."
    )

# Optional: delete deployment first 
try:
    print(f"Routeing 0% traffic to deployment '{DEPLOYMENT_NAME}' before deletion...")
    endpoint = ml_client.online_endpoints.get(name=ENDPOINT_NAME)
    endpoint.traffic = {DEPLOYMENT_NAME: 0}
    ml_client.online_endpoints.begin_create_or_update(endpoint).result()

    print(f"Deleting deployment '{DEPLOYMENT_NAME}' from endpoint '{ENDPOINT_NAME}'...")
    ml_client.online_deployments.begin_delete(
        name=DEPLOYMENT_NAME,
        endpoint_name=ENDPOINT_NAME,
    ).result()
    print("Deployment deleted.")
except ResourceNotFoundError:
    print("Deployment not found (already deleted).")
